# CVD — GEVIS 로드 → 판별 헤드 학습·평가

이 노트북을 위에서부터 실행하면 됩니다. 다른 노트북을 실행할 필요가 없습니다.
저장된 GEVIS 인코더·Fusion을 고정하고 CVD 판별 헤드를 학습합니다.
CVD 전처리·정규화·Run 분할은 기존 방식을 유지합니다.
TRAIN으로 학습하고 Validation으로 epoch를 선택한 뒤 Test를 평가합니다.


In [ ]:
from pathlib import Path
import os, sys
os.environ['HF_HUB_OFFLINE']='1'
os.environ['TRANSFORMERS_OFFLINE']='1'
WORKSPACE_PATH = None  # 데이터·가중치가 다른 폴더에 있으면 해당 경로 지정
SOURCE_ROOT=next((p for p in (Path.cwd(),*Path.cwd().parents)
           if (p/'src/pecvd/gevis.py').is_file()),None)
if SOURCE_ROOT is None:
    raise FileNotFoundError('GEVIS 프로젝트 폴더에서 실행하세요.')
ROOT=SOURCE_ROOT.resolve() if WORKSPACE_PATH is None else Path(WORKSPACE_PATH).expanduser().resolve()
SOURCE=(SOURCE_ROOT/'src').resolve()
if not (ROOT/'config/experiments/gevis_pipeline.json').is_file() or not (ROOT/'artifacts').is_dir():
    raise FileNotFoundError('설정·가중치가 필요합니다. README를 확인하고 WORKSPACE_PATH에 준비한 작업 폴더를 지정하세요.')
for name,module in tuple(sys.modules.items()):
    if name=='pecvd' or name.startswith('pecvd.'):
        for location in [getattr(module,'__file__',None),*getattr(module,'__path__',())]:
            if location and not Path(location).resolve().is_relative_to(SOURCE):
                raise RuntimeError('다른 프로젝트의 pecvd가 로드되어 있습니다. 새 커널로 실행하세요.')
if str(SOURCE) not in sys.path:sys.path.insert(0,str(SOURCE))
import pandas as pd
from pecvd.gevis import load_gevis
from pecvd import gevis_training as training


In [ ]:
TRAINING=training.TrainingConfig(epochs=80,patience=12,batch_size=256,max_seconds=600.)
TARGET_PHASES=None  # None: 전체 공정 경로
GEVIS=load_gevis(ROOT)
display(GEVIS.describe())


In [ ]:
PREPARED=training.prepare_cvd(ROOT,GEVIS)
display(PREPARED.tables['support'])
print({split:len(runs) for split,runs in PREPARED.split_runs.items()})


In [ ]:
RESULTS=training.train_cvd(ROOT,GEVIS,PREPARED,training=TRAINING,steps=TARGET_PHASES)
display(RESULTS.summary)
HISTORY=pd.DataFrame(RESULTS.training['softmax']['history'])
display(HISTORY)
print('저장본:',RESULTS.checkpoints['softmax'])


CVD는 기본 5유형 softmax 판별 헤드를 학습합니다.
헤드는 `artifacts/gevis_heads/pecvd`에 저장합니다.
재사용: `runtime=training.load_head(ROOT, checkpoint, gevis=GEVIS)`.
GEVIS 로드·학습은 Python 라이브러리에서 처리하며 STEP 노트북을 읽거나 실행하지 않습니다.


In [ ]:
if 'ROOT' not in globals() or 'training' not in globals():
    raise RuntimeError('첫 설정 셀부터 실행하세요.')
print('Workspace:', ROOT)


In [ ]:
CVD_CHECKPOINTS = sorted(
    (ROOT / "artifacts/gevis_heads/pecvd").glob("*/answer_head.pt")
)

if not CVD_CHECKPOINTS:
    raise FileNotFoundError("저장된 CVD 판별 헤드가 없습니다.")

for i, path in enumerate(CVD_CHECKPOINTS):
    print(f"[{i}] {path}")

In [ ]:
from types import SimpleNamespace

MODEL_INDEX = 0  # 위 목록에서 기존 평가 모델 번호 선택
CHECKPOINT = CVD_CHECKPOINTS[MODEL_INDEX]

print("평가 입력을 준비합니다.", flush=True)
GEVIS = load_gevis(ROOT)
PREPARED = training.prepare_cvd(ROOT, GEVIS)

RESULTS = SimpleNamespace(
    prepared=PREPARED,
    checkpoints={"softmax": CHECKPOINT},
)
print("시간 측정 준비 완료:", CHECKPOINT)

In [ ]:
import time
import importlib
from collections import Counter

import numpy as np
import pandas as pd
import torch

import pecvd.experiments.ald_head_experiments as head_module
from pecvd import gevis_training as training

# 수정된 CVD 저장본 호환 로더 반영
importlib.reload(head_module)
training = importlib.reload(training)

N_WINDOWS = 20

def sync(device):
    device = torch.device(device)
    if device.type == "cuda":
        torch.cuda.synchronize(device)

# 공정별 Test Window를 번갈아 선택
pools = list(RESULTS.prepared.windows.items())
if not pools:
    raise RuntimeError("평가 입력이 없습니다.")

max_windows = max(len(pool.get("test", [])) for _, pool in pools)
rows = []

for mode, checkpoint in RESULTS.checkpoints.items():
    print(f"[{mode}] 저장 모델을 불러옵니다.", flush=True)

    runtime = training.load_head(
        ROOT,
        checkpoint,
        gevis=GEVIS,
        allow_legacy_cvd=True,
    )
    session = training.Session(runtime)
    device = runtime.module.DEVICE

    def predict(step, w):
        return session.process_window(
            current=w["current"],
            step=step,
            recipe_key=w["cohort"][0],
            loop_key=str(w["cohort"][1]),
            offset_samples=int(w["cohort"][-1]),
            lineage_id=w["lineage_id"],
            run_id=w["run_id"],
            sensor_columns=w["sensor_columns"],
            standardized=True,
        )

    # 지원 여부 확인 및 워밍업 — 측정 시간에서 제외
    selected = []
    excluded = Counter()

    print(f"[{mode}] 평가 Window 선택 및 워밍업", flush=True)
    with torch.inference_mode():
        for i in range(max_windows):
            for step, pool in pools:
                test_windows = pool.get("test", [])
                if i >= len(test_windows):
                    continue

                w = test_windows[i]
                reply = predict(step, w)

                if reply["status"] == "OK":
                    selected.append((step, w))
                else:
                    excluded[reply.get("reason", "unknown")] += 1

                if len(selected) >= N_WINDOWS:
                    break

            if len(selected) >= N_WINDOWS:
                break

    if not selected:
        raise RuntimeError(
            f"{mode}: 추론 가능한 Test Window가 없습니다. "
            f"제외 사유: {dict(excluded)}"
        )

    print(
        f"[{mode}] {len(selected)}개 Window 추론 시간 측정",
        flush=True,
    )

    times = []
    with torch.inference_mode():
        for index, (step, w) in enumerate(selected, 1):
            sync(device)
            start = time.perf_counter()

            reply = predict(step, w)

            sync(device)
            elapsed = time.perf_counter() - start

            if reply["status"] != "OK":
                raise RuntimeError(reply)

            times.append(elapsed)
            print(
                f"  {index}/{len(selected)}: {elapsed:.6f}s",
                flush=True,
            )

    rows.append({
        "방식": mode,
        "측정 Window 수": len(times),
        "총 추론 시간(s)": sum(times),
        "평균(s/Window)": np.mean(times),
        "중앙값(s/Window)": np.median(times),
        "P95(s/Window)": np.percentile(times, 95),
        "평균(ms/Window)": np.mean(times) * 1000,
        "장치": str(device),
    })

INFERENCE_TIMING = pd.DataFrame(rows)
display(INFERENCE_TIMING.round(6))